# Prompt Engineering 101 - Part V.
## Knowledge & Synthesis (RAG)

---

# Module 5: The Knowledge Extension
### *Overcoming Amnesia*

## 1. The Knowledge Cutoff
Base models (like GPT-4 or Gemini) are "frozen in time." They only know what they learned during training (e.g., up to 2023). They do not know today's stock price or your company's new policy.

## 2. RAG (Retrieval Augmented Generation)
RAG is just a fancy term for **"Open Book Testing."**
* **Standard AI:** You ask a question -> AI tries to remember the answer (Hallucination risk).
* **RAG:** You give the AI a document -> AI looks up the answer -> AI writes the response (Fact-based).

## 3. Grounding
"Grounding" means forcing the AI to link every claim to a specific source (a URL, a page number, or a row in a spreadsheet). If it can't find the source, it must say "I don't know."

In [ ]:
# @title 🛠️ Step 1: Laboratory Setup (Gemini API with Wrapper)
!pip install -q -U google-genai

from google.colab import userdata
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# 2. Configure the API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

class GeminiModel:
    def __init__(self, api_key, preferred_model='gemini-2.0-flash'):
        self.client = genai.Client(api_key=api_key)
        self.models = {
            'gemini-2.0-flash': True,
            'gemini-1.5-flash': True,
            'gemini-1.5-flash-8b': True,
        }
        if preferred_model not in self.models:
            self.models = {preferred_model: True, **self.models}
        self.current_model = preferred_model

    def _get_next_available_model(self):
        for model_name, is_available in self.models.items():
            if is_available:
                print(f"🔄 Switching to fallback: {model_name}")
                self.current_model = model_name
                return 
        raise RuntimeError("❌ All available models are currently exhausted.")

    def generate_content(self, contents, config=None):
        try:
            return self.client.models.generate_content(
                model=self.current_model,
                contents=contents,
                config=config
            )
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"⚠️ Quota exceeded for {self.current_model}. Switching...")
                self.models[self.current_model] = False
                self._get_next_available_model()
                return self.generate_content(contents, config)
            raise e

try:
    model = GeminiModel(GEMINI_API_KEY, preferred_model='gemini-2.0-flash')
    print(f"✅ Connection Established. Using {model.current_model}.")
except Exception as e:
    print(f"❌ Error: {e}")

---

### **Phase 1: Manual RAG (Context Stuffing)**

In [ ]:
# @title 📄 Topic 1: The Context Window (Manual RAG)
# Concept: The simplest form of RAG is just pasting the text into the prompt.
# Gemini 1.5/2.0 has a massive context window (1 Million+ tokens), so we can paste whole books.

# SCENARIO: A new (fake) law just passed. The AI doesn't know about it yet.
fake_law_text = """
THE 'NO-EMAIL FRIDAY' ACT OF 2025
Section 1: It shall be illegal for any employer to send electronic mail 
to employees between the hours of 00:01 and 23:59 on Fridays.
Section 2: Violations are punishable by a fine of $500 per email.
Section 3: Exceptions are granted for emergency services (Police, Fire, Hospitals).
"""

# 1. Ask WITHOUT the text (The AI will likely say this law doesn't exist)
print("--- WITHOUT CONTEXT ---")
try:
    print(model.generate_content("What is the penalty under the No-Email Friday Act?").text)
except:
    print("AI: I don't know that law.")

# 2. Ask WITH the text (Manual RAG)
rag_prompt = f"""
SOURCE DOCUMENT:
{fake_law_text}

QUESTION:
Based ONLY on the document above, can a Hospital send an email on Friday?
Cite the section number.
"""

print("\n--- WITH CONTEXT (RAG) ---")
print(model.generate_content(rag_prompt).text)

In [ ]:
# @title 🏢 LAB 1: The Policy Analyst
# SCENARIO: You are an HR Manager. Employees keep asking about the confusing "Remote Work Policy".
# TASK: Create a "Policy Bot" that answers questions based on the text below.

policy_text = """
REMOTE WORK POLICY (REV 2024)
1. Eligibility: Employees must work at the company for >6 months.
2. Core Hours: You must be online from 10am to 2pm EST.
3. Equipment: The company provides a laptop. Monitors are reimbursed up to $200.
4. Location: You may work from any state, but International work requires VP approval.
"""

# --- STUDENT WORKSPACE ---
question = "Can I move to France and work from there?"

# Write a prompt that uses the policy_text to answer the question.
# Constraint: If the policy doesn't say, answer "Contact HR".
my_prompt = f"""
POLICY:
{policy_text}

USER QUESTION: {question}

INSTRUCTION:
Answer based strictly on the policy. 
"""
# -------------------------

display(Markdown(model.generate_content(my_prompt).text))

---

### **Phase 2: Tool Use (The Web)**

In [ ]:
# @title 🌍 Topic 2: Google Search Grounding (Real-Time Data)
# Concept: We can give the AI a "Tool" (Google Search) to look up facts it doesn't know.
# Note: In the Gemini API, we enable this via 'tools'.

from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# We create a config that enables Google Search
search_tool = Tool(google_search=GoogleSearch())
search_config = GenerateContentConfig(tools=[search_tool])

# Question about something that happened RECENTLY (AI training data won't have it)
# e.g., The outcome of a recent sports game or stock price.
current_question = "What is the current stock price of NVIDIA (NVDA) right now?"

print(f"Asking: {current_question}...")
response = model.generate_content(current_question, config=search_config)

print("--- LIVE RESPONSE ---")
print(response.text)

# Check grounding metadata (sources)
# Note: The API returns metadata about where it found the info.
if response.candidates[0].grounding_metadata.search_entry_point:
    print("\n[Source: Google Search Data Used]")

In [ ]:
# @title 📉 LAB 2: The Market Researcher
# TASK: Find the latest news on a specific topic that happened THIS WEEK.
# (If we didn't use the tool, the AI would hallucinate or say "I'm cut off in 2023").

# --- STUDENT WORKSPACE ---
topic = "What are the latest rumors about the iPhone 17 release date?"

# Use the search_config we defined above
response = model.generate_content(topic, config=search_config)
# -------------------------

display(Markdown(response.text))

---

### **Phase 3: Data Analysis (The "Code Interpreter")**

In [ ]:
# @title 📊 Topic 3: RAG with Data (CSV Analysis)
# Concept: RAG isn't just for words; it's for numbers.
# We can paste a CSV string into the context and ask the AI to analyze it.

# A messy sales report (CSV format)
csv_data = """
Date,Region,Product,Sales,Units
2024-01-01,North,Widget A,1000,10
2024-01-02,South,Widget A,500,5
2024-01-03,North,Widget B,2000,20
2024-01-04,North,Widget A,1200,12
2024-01-05,South,Widget B,0,0
"""

analysis_prompt = f"""
DATASET:
{csv_data}

TASK:
1. Act as a Data Scientist.
2. Calculate the Total Sales for the 'North' region.
3. Identify which product is the best seller by revenue.
4. Output the result as a Markdown summary.
"""

display(Markdown(model.generate_content(analysis_prompt).text))

In [ ]:
# @title 🐍 Topic 4: Code Execution (The Calculator)
# Concept: LLMs are bad at math (System 1). But they are great at writing Python (System 2).
# We can ask the AI to "Write and Run code" to solve data problems accurately.

# We simulate this by asking for the code, then (conceptually) running it.
# (Gemini API has a Code Execution tool, but for this beginner class, we just generate the code).

math_prompt = f"""
DATA:
{csv_data}

TASK:
Write a Python script using pandas to:
1. Load this data.
2. Calculate the average sales per unit.
3. Plot a bar chart of Sales by Region.
"""

print(model.generate_content(math_prompt).text)

In [ ]:
# @title 🔬 LAB 3: The Data Scientist
# SCENARIO: You have a list of employee feedback scores.
# TASK: Find the average score and the most common complaint.

feedback_csv = """
EmployeeID,Score,Comment
101,2,The coffee is cold.
102,5,I love the new office!
103,1,The coffee is terrible and cold.
104,4,Great team atmosphere.
105,2,Need better coffee.
"""

# --- STUDENT WORKSPACE ---
# Write a prompt to analyze the CSV.
# Ask for: Average Score AND a summary of the "Coffee Issue".
data_prompt = f"""
DATA:
{feedback_csv}

Analyze the data:
1. Calculate average score.
2. Summarize the qualitative feedback themes.
"""
# -------------------------

display(Markdown(model.generate_content(data_prompt).text))

---

### **Phase 4: Multi-Modal RAG (Images)**

In [ ]:
# @title 🖼️ Topic 5: Vision RAG (Asking questions about images)
# Concept: "Reading" isn't just text. Gemini can "read" charts, screenshots, and photos.
# Since we can't easily upload files in this simple Colab setup, we will simulate the prompt structure.

# (Instructor Note: Explain that in the real world, you'd upload the image file).

vision_prompt = """
[IMAGE UPLOADED: A screenshot of a complex Excel Dashboard]

TASK:
Look at the chart in the top right corner (Revenue Q3).
Extract the number for 'September' and tell me if it is higher than 'August'.
"""

print("--- SIMULATION ---")
print("Since we cannot upload a file here easily, imagine the AI replies:")
print("'Based on the bar chart, September revenue is $50k, which is 10% higher than August ($45k).'")

# Discussion: This effectively turns any image into a database you can query.

---

# 🏠 Homework: The Due Diligence Bot

### The Scenario
You are analyzing a startup for potential investment.
You have two sources of information:
1.  **The Pitch Deck (Text):** Claims they are the market leader.
2.  **The News (Search):** Real-world articles about their competitors.

### The Task
Write a Python script using our `model` wrapper that:
1.  **Ingests** a "Fake Pitch Deck" (text string) claiming they have 0 competitors.
2.  **Uses Google Search Tool** to find real competitors for "AI-powered Toaster".
3.  **Synthesizes** a report highlighting the discrepancy between the Pitch (0 competitors) and Reality (Search Results).

### Submission
Submit the Python code and the final "Risk Report".

In [ ]:
# YOUR CODE GOES HERE